# Build and Start MCP Servers

In this notebook, we start the 4 MCP servers, verify they are running,
and inspect their available tools via the MCP protocol.

## Learning Objectives
- Start MCP servers as Python subprocesses
- Verify server health via the `tools/list` MCP endpoint
- Understand each server's tool signatures
- Confirm readiness for the harness controller

In [ ]:
import os
import sys
import time
import socket
import subprocess

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'), override=True)

print("\u2705 Environment loaded")

## 1. Start MCP Servers

Each MCP server runs as a FastMCP process with streamable-http transport.
We start them as subprocesses and wait for port readiness.

In [ ]:
MCP_SERVERS = [
    ("vector-search-mcp", "mcp_servers.vector_search_mcp.server", 9002),
    ("web-search-mcp", "mcp_servers.web_search_mcp.server", 9003),
    ("verification-mcp", "mcp_servers.verification_mcp.server", 9004),
    ("observability-mcp", "mcp_servers.observability_mcp.server", 9005),
]

def port_in_use(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

processes = []
for name, module, port in MCP_SERVERS:
    if port_in_use(port):
        print(f"\u2705 {name} already running on port {port}")
        continue
    proc = subprocess.Popen(
        [sys.executable, "-m", module],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        cwd=os.path.join(os.getcwd(), ".."),
    )
    processes.append((name, port, proc))
    print(f"\U0001f680 Started {name} (pid={proc.pid}) on port {port}")

# Wait for all servers to be ready
max_wait = 15
for elapsed in range(max_wait):
    time.sleep(1)
    pending = [n for n, p, _ in processes if not port_in_use(p)]
    if not pending:
        break

if pending:
    print(f"\u26a0\ufe0f Servers not ready after {max_wait}s: {pending}")
else:
    print(f"\n\u2705 All 4 MCP servers ready ({elapsed+1}s)")

## 2. Verify Server Health via MCP Protocol

Each MCP server exposes `tools/list` at its `/mcp/` endpoint.
We query this to confirm the server is responding and list available tools.

In [ ]:
import httpx
import json

async def list_mcp_tools(base_url: str) -> list[dict]:
    """Query a MCP server's tools/list endpoint."""
    payload = {
        "jsonrpc": "2.0",
        "method": "tools/list",
        "id": "1",
        "params": {},
    }
    async with httpx.AsyncClient(timeout=10) as client:
        resp = await client.post(f"{base_url}/mcp/", json=payload)
        result = resp.json()
        return result.get("result", {}).get("tools", [])

print("\U0001f50d Querying MCP servers for available tools...\n")

for name, _, port in MCP_SERVERS:
    url = f"http://127.0.0.1:{port}"
    try:
        tools = await list_mcp_tools(url)
        print(f"\u2705 {name} ({url}) — {len(tools)} tools:")
        for tool in tools:
            params = list(tool.get("inputSchema", {}).get("properties", {}).keys())
            print(f"     \u2022 {tool['name']}({', '.join(params)})")
        print()
    except Exception as e:
        print(f"\u274c {name} ({url}) — ERROR: {e}\n")

## 3. MCP Server Architecture

Each server follows the same pattern using `FastMCP`:

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("server-name", stateless_http=True)

@mcp.tool()
def my_tool(query: str, top_k: int = 5) -> list[dict]:
    """Tool description shown to clients."""
    # Implementation...
    return results

if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="0.0.0.0", port=9002)
```

**Key design decisions:**
- `stateless_http=True` — No session state at the MCP layer (state lives in the orchestrator)
- Transport: `streamable-http` — Standard HTTP POST, works with any MCP client
- Each server is independently deployable as a container

## 4. Server Source Code Paths

| MCP Server | Module | Port | Source |
|------------|--------|------|--------|
| vector-search-mcp | `mcp_servers.vector_search_mcp.server` | 9002 | `mcp_servers/vector_search_mcp/server.py` |
| web-search-mcp | `mcp_servers.web_search_mcp.server` | 9003 | `mcp_servers/web_search_mcp/server.py` |
| verification-mcp | `mcp_servers.verification_mcp.server` | 9004 | `mcp_servers/verification_mcp/server.py` |
| observability-mcp | `mcp_servers.observability_mcp.server` | 9005 | `mcp_servers/observability_mcp/server.py` |

**Not MCP-wrapped** (by design):
- Document ingestion (Docling) — backend handles directly
- Query rewriting, planning, drafting — direct LLM calls in orchestrator

## 5. Cleanup (Optional)

Stop the MCP servers started in this notebook.
Skip this if you plan to continue with notebook 3 (Test MCP Tools).

In [ ]:
# Uncomment to stop servers started in this notebook:
# for name, port, proc in processes:
#     proc.terminate()
#     print(f"Stopped {name} (pid={proc.pid})")

print("\U0001f4a1 MCP servers still running — ready for notebook 3 (Test MCP Tools)")